<a href="https://colab.research.google.com/github/syhennie/data-quality-process/blob/main/features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Настройка окружения

In [1]:
%pip install numpy gensim pandas scipy sdv matplotlib graphviz faker

     ---------------------------------------- 0.0/60.9 kB ? eta -:--:--
     --------------------------------- ------ 51.2/60.9 kB 2.6 MB/s eta 0:00:01
     ---------------------------------------- 60.9/60.9 kB 3.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ---------------------------------------- 60.8/60.8 kB ? eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB 3.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/115.4 kB ? eta -:--:--
     -------------------------------------- 115.4/115.4 kB 6.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ---- ----------------------------------- 1.5/13.1 MB 46.9 MB/s eta 0:00:01
   -------- ------------------------------- 2.9/13.1 MB 30.9 MB/s eta 0:00:01
   ------------------- -------------------- 6.5/13.1 MB 46.0 MB/s eta 0:00:01
   -----------


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sdv
import pathlib

In [2]:
df = pd.read_csv(pathlib.Path('../test.csv'))
df = df.iloc[:, 1:]
df.head()

,article,highlights
0,Ever noticed how plane seats appear to be gett...,Experts question if packed out planes are put...
1,A drunk teenage boy had to be rescued by secur...,Drunk teenage boy climbed into lion enclosure ...
2,Dougie Freedman is on the verge of agreeing a ...,Nottingham Forest are close to extending Dougi...
3,Liverpool target Neto is also wanted by PSG an...,Fiorentina goalkeeper Neto has been linked wit...
4,Bruce Jenner will break his silence in a two-h...,"Tell-all interview with the reality TV star, 6..."


In [37]:
from gensim.models import Word2Vec, FastText
from gensim.utils import simple_preprocess

model = FastText.load("../fasttext_lee_background")

MAX_TOKENS_PER_ENTRY = 10
VECTOR_SIZE = 64

def vectorise_entries(entries):
    features = []
    for entry in entries:
        tokens = simple_preprocess(entry)
        vectors = [model.wv[token] for token in tokens]
        length = min(len(vectors), MAX_TOKENS_PER_ENTRY)
        trimmed_vectors = vectors[:length]
        if length < MAX_TOKENS_PER_ENTRY:
            padding_vectors = [np.zeros(VECTOR_SIZE) for _ in range(MAX_TOKENS_PER_ENTRY - length)]
            trimmed_vectors += padding_vectors
        features.append(np.concatenate(trimmed_vectors))
    return np.array(features)

In [38]:
def decode_row(row):
    entry = ""
    for i in range(MAX_TOKENS_PER_ENTRY):
        vectorised_token = row[i * VECTOR_SIZE:(i + 1) * VECTOR_SIZE]
        most_similar_words = model.wv.similar_by_vector(vectorised_token, topn=5)
        for word, _ in most_similar_words:
            if str(word).isalnum():
                entry += f" {word}"
    return entry

In [39]:
import torch
print(torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
for i in range(num_gpus):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
torch.cuda.set_device(0)

True
Device 0: NVIDIA GeForce RTX 5070 Ti


In [40]:
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer


df_sample = df.sample(1000)
df_result = pd.DataFrame()

for column in df_sample.columns:
    entries = [x for x in df_sample[column]]
    vectorised_entries = vectorise_entries(entries)

    column_df = pd.DataFrame(vectorised_entries)
    metadata = Metadata.detect_from_dataframe(column_df)
    synthesizer = CTGANSynthesizer(metadata)
    synthesizer.fit(column_df)

    synthetic_data = synthesizer.sample(num_rows=1000).values
    result = np.apply_along_axis(decode_row, 1, synthetic_data)
    df_result[column] = result

df_result
    

c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
0                      11
1                      11
2                      11
3                      11
4                      11
5                      11
6                      11
7                      11
8                      11
9                      11
10                     11
11                     11
12                     11
13                     11
14                     11
15                     11
16                     11
17                     11
18                     11
19                     11
20                     11
21                     11
22                     11
23                     11
24                     11
25                     11
26                     11
27                     11
28                     11
29                     11
30                     11


c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
0                      11
1                      11
2                      11
3                      11
4                      11
5                      11
6                      11
7                      11
8                      11
9                      11
10                     11
11                     11
12                     11
13                     11
14                     11
15                     11
16                     11
17                     11
18                     11
19                     11
20                     11
21                     11
22                     11
23                     11
24                     11
25                     11
26                     11
27                     11
28                     11
29                     11
30                     11


c:\Users\stepa\AppData\Local\Programs\Python\Python311\Lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


,article,highlights
0,25 drug HIV 15 Sarah 2 21 UN Ms 18 Mt 21 Naur...,Ms 2 48 data swept HIV 25 48 drug SES 25 150 ...
1,Ms Dr 10 boy Assa No Up 48 keep 12 80 If 150 ...,HIV 48 No 25 AFP 80 As If 13 14 UN Kabul pilo...
2,12 Road keep nine Up No SES my 23 AFP Blue Up...,SES No 25 UN 25 keep 150 15 Sir Up 150 big Dr...
3,UN Assa drug As No As 1 ASIO 13 my Assa embas...,23 25 If 11 1 40 UN blaze At 400 UN Dr On 150...
4,26 24 If An Ms Ms Sir happy swept 24 blaze At...,80 5 24 drug Nauru 26 1 Up UN Sir 2 UN 18 run...
...,...,...
995,Up SES On pilot 2002 40 Ms HIH gun 500 23 Ms ...,25 On No 80 ASIO SES As HIV 14 25 W 24 15 1 A...
996,25 drug Karzai am 20 Ms 21 run 14 18 24 Up 20...,1 blaze 80 Al Nauru 13 40 18 Dr keep 13 23 15...
997,SES HIV Mt Sarah crowd UN At Kabul 14 If On S...,23 5 25 13 18 2 25 No UN As UN No 150 48 Sara...
998,13 drug Karzai vote W At Lee 14 if Up 48 oil ...,25 23 150 11 48 1 As 26 UN Karzai 25 26 18 14...


In [41]:
def get_vectorised_entries(entries):
    features = []
    for entry in entries:
        tokens = simple_preprocess(entry, min_len=1)
        vectors = [model.wv[token] for token in tokens]
        features.append(np.mean(vectors, axis=0))
    return np.array(features)

In [42]:
summary_stats = []
features_data = []

for column in df_sample.columns:
    entries = [x for x in df_sample[column]]
    features = get_vectorised_entries(entries)
    vectorised_entries = vectorise_entries(entries)

    # Features-based metrics
    #
    # mean = np.mean(features, axis=0)
    # standard_deviation = np.std(features, axis=0)
    # median = np.median(features, axis=0)
    # asymmetry = stats.skew(features, axis=0)
    # excess = stats.kurtosis(features, axis=0)

    mean = np.mean(vectorised_entries, axis=0)
    standard_deviation = np.std(vectorised_entries, axis=0)
    median = np.median(vectorised_entries, axis=0)
    asymmetry = stats.skew(vectorised_entries, axis=0)
    excess = stats.kurtosis(vectorised_entries, axis=0)

    summary_stats.append({
        'column': column,
        'overall_mean': np.mean(mean),
        'overall_std': np.mean(standard_deviation),
        'std_of_means': np.std(mean),
        'mean_of_medians': np.mean(median),
        'asymmetry_avg': np.mean(asymmetry),
        'excess_avg': np.mean(excess),
        'n_entries': len(entries),
        'vector_dim': vectorised_entries.shape[1]
    })

    features_data.append({
            'column': column,
            'vectors': features
        })

summary_df = pd.DataFrame(summary_stats)
summary_df

,column,overall_mean,overall_std,std_of_means,mean_of_medians,asymmetry_avg,excess_avg,n_entries,vector_dim
0,article,-0.033450,0.189811,0.418471,-0.03136,-0.063171,-0.133177,1000,640
1,highlights,-0.031793,0.194116,0.400827,-0.02884,-0.087595,0.238360,1000,640


In [43]:
def validate_synthetic_data(original_stats, synthetic_data):
    validation_results = []

    for column in synthetic_data.columns:
        column_stats = original_stats[original_stats['column'] == column]
        entries = [x for x in synthetic_data[column]]
        features = get_vectorised_entries(entries)

        synth_mean = np.mean(features, axis=0)
        synth_std = np.std(features, axis=0)
        synth_skew = stats.skew(features, axis=0)
        synth_kurt = stats.kurtosis(features, axis=0)

        validation_results.append({
            'column': column_stats['column'].tolist()[0],
            'original_mean': column_stats['overall_mean'].tolist()[0],
            'synthetic_mean': np.mean(synth_mean),
            'mean_error': np.abs(np.mean(synth_mean) - column_stats['overall_mean'].tolist()[0]),

            'original_std': column_stats['overall_std'].tolist()[0],
            'synthetic_std': np.mean(synth_std),
            'std_error': np.abs(np.mean(synth_std) - column_stats['overall_std'].tolist()[0]),

            'original_skew': column_stats['asymmetry_avg'].tolist()[0],
            'synthetic_skew': np.mean(synth_skew),
            'skew_error': np.abs(np.mean(synth_skew) - column_stats['asymmetry_avg'].tolist()[0]),

            'original_kurt': column_stats['excess_avg'].tolist()[0],
            'synthetic_kurt': np.mean(synth_kurt),
            'kurt_error': np.abs(np.mean(synth_kurt) - column_stats['excess_avg'].tolist()[0]),

            'n_samples': len(features),
            'vector_dim': features.shape[1]
        })

    return pd.DataFrame(validation_results)

In [45]:
validation_df = validate_synthetic_data(summary_df, df_result)

validation_df[['column', 'mean_error', 'std_error', 'skew_error', 'kurt_error']]
# print(validation_df[['column', 'mean_error', 'std_error', 'skew_error', 'kurt_error']])

,column,mean_error,std_error,skew_error,kurt_error
0,article,0.015515,0.150667,0.044555,0.022497
1,highlights,0.011596,0.152627,0.038436,0.391124
